# PyPSA-Eur style power system plot

## Authored by F.Hofmann

The notebook reproduces the plot like a beautiful scheme of the
European Transmission System published in https://arxiv.org/abs/1806.01613.

In [62]:
#"""
#Created on Mon Sep 19 15:51:31 2022
#
#@author: fabian
#"""

In [ ]:
import pypsa
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
from datetime import datetime
from cartopy import crs as ccrs
from pypsa.plot import add_legend_circles, add_legend_lines, add_legend_patches
import plotly.express as px
import hvplot.pandas
import matplotlib.colors as mcolors
import os

Two files are needed:
* PyPSA network file (e.g. "elec.nc" contains a lot of details and looks perfect)
* a country shape file (may by found in "resources/shapes/country_shapes.geojson")

In [ ]:
scenario_name = "Kenya-2030-Sec-Great-Rift-1"  # scenario name, default value is "" for tutorial or default configuration
                    # value shall be non null if a scenario name is specified under the "run" tag in the config file

scenario_subpath = scenario_name + "/" if scenario_name else ""

#export_level = 10
export_level = 0

PARENT = os.path.realpath("/mnt/e/Github-Alex/pypsa-earth/") + "/"

n=pypsa.Network(PARENT + f"/results/" + scenario_name + f"/postnetworks/elec_s_31_ec_lcopt_Co2L0.68_144H_2030_0.09_AB_{export_level}export.nc")

regions_onshore = gpd.read_file(PARENT + "/resources/" + scenario_name + "/shapes/country_shapes.geojson")

In [92]:
OUTPUT = f"allcarrier_{export_level}export.pdf"
OUTPUT_png = f"allcarrier_{export_level}export.png"

Define plot parameters:

In [93]:
bus_scale = 5e3 
line_scale = 8e2
capacity_scale = 10

In [94]:
extent = [33.5, 42, -5, 5] # for country NZ [165.0, 180.0, -50.0, -25.0] 
parameter = "p_nom_opt"

In [102]:
# Select only certain carriers, in this case "gas" is excluded. Note; the carrier color must be present in n.carriers.color
carrier_sel = ['geothermal', 'onwind', 'solar', 'ror', 'CCGT', 'OCGT', 'coal',
       'offwind-ac', 'offwind-dc', 'oil']

In [ ]:
gen = n.generators[n.generators.carrier.isin(carrier_sel)].groupby(["bus", "carrier"])[parameter].sum()

In [98]:
# Select only certain carriers, in this case "gas" is excluded. Note; the carrier color must be present in n.carriers.color
carrier_sel = ['geothermal', 'onwind', 'solar', 'ror', 'CCGT', 'OCGT', 'coal',
       'offwind-ac', 'offwind-dc', 'oil']

In [99]:
gen = n.generators[n.generators.carrier.isin(carrier_sel)].groupby(["bus", "carrier"])[parameter].sum()
sto = n.storage_units.groupby(["bus", "carrier"])[parameter].sum()
buses = pd.concat([gen, sto])
#buses.name = "p_nom"

In [ ]:
# fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": ccrs.EqualEarth(), "frameon": False})

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": ccrs.EqualEarth(n.buses[n.buses.carrier == "AC"].x.mean())})

with plt.rc_context({"patch.linewidth": 0.}):
    n.plot(
        bus_sizes=buses / bus_scale,
        bus_alpha=0.7,
        line_widths=n.lines.s_nom / line_scale,
        link_widths=n.links.p_nom / line_scale,
        line_colors="teal",
        ax=ax,
        margin=0.2,
        color_geomap=None,
    )

regions_onshore.plot(
    ax=ax,
    facecolor="whitesmoke",
    edgecolor="white",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0,
)

ax.set_extent(extent)
legend_kwargs = {"loc": "upper left", "frameon": False}
# circles legend may requite some fine-tuning
legend_circles_dict = {"bbox_to_anchor": (1, 0.6), "labelspacing": 2.5, **legend_kwargs}
bus_sizes = [100, 1e3]  # in MW
line_sizes = [500, 2000]  # in MW
add_legend_circles(
    ax,
    [s / bus_scale for s in bus_sizes],
    [f"{s / 1000} GW" for s in bus_sizes],
    legend_kw=legend_circles_dict,    
)
add_legend_lines(
    ax,
    [s / line_scale for s in line_sizes],
    [f"{s / 1000} GW" for s in line_sizes],
    legend_kw={"bbox_to_anchor": (1, 0.8), **legend_kwargs},
)
add_legend_patches(
    ax,
    n.carriers.loc[carrier_sel].color,
    n.carriers.loc[carrier_sel].nice_name,
    legend_kw={"bbox_to_anchor": (1, 0), **legend_kwargs, "loc":"lower left"},
)


fig.tight_layout()
fig.savefig(OUTPUT, bbox_inches="tight", dpi=300)
fig.savefig(OUTPUT_png, bbox_inches="tight", dpi=300)

In [ ]:
optimal_capacity = n.statistics.optimal_capacity(comps=["Generator"]).droplevel(0).div(1e3)
installed_capacity = n.statistics.installed_capacity(comps=["Generator"]).droplevel(0).div(1e3)
generation_capacity_expansion = optimal_capacity - installed_capacity
generation_capacity_expansion.drop(["load"], inplace=True)
generation_capacity_expansion.plot.bar(title="Generator capacity expansion in GW")

In [ ]:
n.loads

In [ ]:
n.generators

In [ ]:
n.generators

In [ ]:
import yaml

PARENT = os.path.realpath("/mnt/e/Github-Alex/pypsa-earth/") + "/"
config = yaml.safe_load(open(PARENT + "config.yaml"))

colors = {key.lower(): value.lower() for key, value in config["plotting"]["tech_colors"].items()}
nice_names = {value.lower(): key for key, value in config["plotting"]["nice_names"].items()}

rename_cols = {
    '-': 'Load',
    'load': 'load shedding',
}

energy_balance = (
    n.statistics.energy_balance()
    .loc[:, :, "AC"]
    .groupby("carrier")
    .sum()
    .div(1e6)
    .to_frame()
    .T
    .rename(columns=rename_cols)
)

# color-matching
color_list = []
for col in energy_balance.columns:
    original_name = col.lower()
    key_name = nice_names.get(original_name, original_name)
    color = colors.get(key_name.lower(), 'gray')
    color_list.append(color)


fig, ax = plt.subplots()
energy_balance.plot.bar(stacked=True, ax=ax, title="Energy Balance in TWh", color=color_list)
handles, labels = ax.get_legend_handles_labels()
nice_labels = [config["plotting"]["nice_names"].get(label, label) for label in energy_balance.columns]
ax.legend(handles, nice_labels, bbox_to_anchor=(1, 0), loc="lower left", title=None, ncol=1)

plt.show()

In [ ]:
n.generator_t.sum

In [ ]:
pd.set_option('display.max_columns', None)
energy_balance

In [ ]:
energy_balance